# 🛠️ Лекція 3 — Інженерія LLM-агентів

## ⚙️ Встановлення залежностей

> **⚠️ API-ключі:** Для секцій з використанням моделей OpenAI потрібно встановити змінні середовища `OPENAI_API_KEY`. Опціонально, та/або `ANTHROPIC_API_KEY`, якщо хочете перевірити роботу LangChain з Claude.


In [ ]:
!pip install -q "langchain>=1.2.12" langchain-openai langchain-anthropic langchain-community langchain-core langchain-chroma requests
!pip install -q chromadb pydantic langgraph

In [ ]:
import os
import json
import subprocess
import sys
import requests

os.environ["OPENAI_API_KEY"] = "..."
#os.environ["ANTHROPIC_API_KEY"] = ""

---

## 1. Tool / Function Calling

### 🔑 Що таке Tool Calling?

**Tool Calling** — механізм, через який LLM взаємодіє із зовнішнім світом:
- LLM генерує **структурований запит** (JSON)
- Ваш код **виконує відповідну дію**
- Результат **повертається в LLM**

### Еволюція підходу

| Період | Підхід |
|--------|--------|
| 2022 | Парсинг регулярками |
| 2023 | Function Calling (OpenAI) |
| 2023-24 | Tool Use як стандарт індустрії |
| 2024+ | Structured Outputs |

### Як це працює (6 кроків)

1. **User Query** — користувач надсилає запит
2. **LLM + Tools** — LLM аналізує запит з описами tools у контексті
3. **Tool Call JSON** — `{"name": "get_weather", "args": {"city": "Kyiv"}}`
4. **Execution** — ваш код виконує реальну функцію
5. **Result → LLM** — результат повертається в контекст LLM
6. **Final Response** — LLM формує фінальну відповідь користувачу


### 📐 Tool Schema — JSON опис інструменту

Кожен інструмент описується трьома ключовими полями:
- **name** — унікальний ідентифікатор
- **description** — LLM вибирає tool саме за цим (пишіть чітко!)
- **parameters** — JSON Schema з типом і описом кожного аргументу


In [ ]:
# Tool Schema — what the LLM sees when deciding which tool to call

weather_tool_schema = {
    "type": "function",
    "name": "get_weather",
    "description": "Get current weather for a city. Returns temperature in Celsius and conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'Kyiv', 'London'"
            }
        },
        "required": ["city"]
    }
}

print("Tool Schema:")
print(json.dumps(weather_tool_schema, indent=2))

Tool Schema:
{
  "type": "function",
  "name": "get_weather",
  "description": "Get current weather for a city. Returns temperature in Celsius and conditions.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. 'Kyiv', 'London'"
      }
    },
    "required": [
      "city"
    ]
  }
}


### 🔧 Tool Calling через OpenAI Responses API

Три кроки виклику інструментів через API:

1. **Крок 1** — визначаємо реальну Python-функцію та передаємо `tools=` до LLM
2. **Крок 2** — модель повертає `tool_calls` з назвою функції та аргументами
3. **Крок 3** — ми розпаковуємо аргументи і викликаємо реальну функцію


In [ ]:
from openai import OpenAI

client = OpenAI()

# Step 0: Define your real Python function — calls wttr.in (free, no API key)
def get_weather(city: str) -> str:
    """Get current weather for a city using wttr.in API."""
    try:
        resp = requests.get(
            f"https://wttr.in/{city}",
            params={"format": "j1"},
            headers={"User-Agent": "lecture-demo"},
            timeout=5,
        )
        resp.raise_for_status()
        current = resp.json()["current_condition"][0]
        return json.dumps({
            "city": city,
            "temperature": int(current["temp_C"]),
            "conditions": current["weatherDesc"][0]["value"],
            "humidity": f"{current['humidity']}%",
        })
    except Exception as e:
        return json.dumps({"city": city, "error": str(e)})

# Tool registry — maps tool names to Python functions
tool_registry = {"get_weather": get_weather}

# Step 1: Define tools and call the API
tools = [weather_tool_schema]

response = client.responses.create(
    model="gpt-5-mini",
    input=[
        {"role": "system", "content": "You are a weather assistant. Use the get_weather tool."},
        {"role": "user", "content": "What is the weather in Kyiv?"},
    ],
    tools=tools,
)

# Step 2: Extract tool call from response
tool_call = next(item for item in response.output if item.type == "function_call")
print(f"Tool called:  {tool_call.name}")
print(f"Arguments:    {tool_call.arguments}")

# Step 3: Dispatch to the right function via registry
func = tool_registry[tool_call.name]
args = json.loads(tool_call.arguments)
result = func(**args)
print(f"Result:       {result}")

Tool called:  get_weather
Arguments:    {"city":"Kyiv"}
Result:       {"city": "Kyiv", "temperature": 4, "conditions": "Sunny", "humidity": "71%"}


### 🔄 Повний цикл: tool call → результат → фінальна відповідь

Після виконання функції ми повертаємо результат у LLM, щоб він сформував людську відповідь.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Tokyo and London?"},
]

response = client.responses.create(
    model="gpt-5-mini",
    input=messages,
    tools=tools,
)

# Tool registry — maps tool names to Python functions
tool_registry = {
    "get_weather": get_weather,
}

# Execute all tool calls and collect results
followup_input = messages.copy()
followup_input.extend(response.output)

for item in response.output:
    if item.type == "function_call":
        func = tool_registry.get(item.name)
        if func is None:
            raise ValueError(f"Unknown tool: {item.name}")
        args = json.loads(item.arguments)
        result = func(**args)
        print(f"🔧 {item.name}({args}) → {result}")
        followup_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })

# Feed results back to get final human-readable response
final_response = client.responses.create(
    model="gpt-5-mini",
    input=followup_input,
    tools=tools,
)

final_text = next(item for item in final_response.output if item.type == "message").content[0].text
print(f"\n💬 Final response: {final_text}")

🔧 get_weather({'city': 'Tokyo'}) → {"city": "Tokyo", "temperature": 11, "conditions": "Partly cloudy", "humidity": "47%"}
🔧 get_weather({'city': 'London'}) → {"city": "London", "temperature": 8, "conditions": "Overcast", "humidity": "81%"}

💬 Final response: Here’s the current weather:

- Tokyo: 11°C (≈52°F), partly cloudy, humidity 47%.
- London: 8°C (≈46°F), overcast, humidity 81%.

Want a short-term forecast, hourly details, or weather for any other cities?


---

## 2. Structured Output — три підходи

LLM за замовчуванням повертає текст. Для інтеграції з кодом потрібні **структуровані дані**. Є три основних підходи:

| Підхід | Опис | Гарантія схеми |
|--------|------|----------------|
| **JSON Mode** | Просимо модель відповідати JSON | Слабка |
| **Function Calling** | Модель заповнює параметри tool schema | Так |
| **Pydantic + Structured Outputs** | Валідація через Python-моделі | Повна |


In [ ]:
# Approach 1: JSON Mode
response = client.responses.create(
    model="gpt-5-mini",
    text={"format": {"type": "json_object"}},
    input=[{
        "role": "user",
        "content": "Return a JSON object with fields: name (string), age (integer), city (string) for a fictional person from Kyiv."
    }]
)

data = json.loads(response.output_text)
print("JSON Mode result:", json.dumps(data, indent=2))
print(f"Type: {type(data)}")

JSON Mode result: {
  "name": "Anastasiya Koval",
  "age": 29,
  "city": "Kyiv"
}
Type: <class 'dict'>


In [ ]:
# Approach 2: Function Calling (schema-constrained via tools)
# The tool schema constrains the model's output format

extract_tool = {
    "type": "function",
    "name": "extract_person",
    "description": "Extract person information from text",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "Person's full name"},
            "age": {"type": "integer", "description": "Person's age"},
            "city": {"type": "string", "description": "City of residence"},
            "skills": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of skills"
            }
        },
        "required": ["name", "age", "city", "skills"]
    }
}

response = client.responses.create(
    model="gpt-5-mini",
    input=[{
        "role": "user",
        "content": "Extract info: Nazar is a 25-year-old Python developer from Kyiv who also knows TypeScript and ML."
    }],
    tools=[extract_tool],
    tool_choice="required"
)


tool_call = next(item for item in response.output if item.type == "function_call")
person_data = json.loads(tool_call.arguments)


print("Function Calling result:", json.dumps(person_data, indent=2))

Function Calling result: {
  "name": "Nazar",
  "age": 25,
  "city": "Kyiv",
  "skills": [
    "Python",
    "TypeScript",
    "ML"
  ]
}


In [ ]:
# Approach 3: Pydantic — full validation in Python
from pydantic import BaseModel, Field, ValidationError

class Person(BaseModel):
    name: str = Field(description="Person's full name")
    age: int = Field(ge=0, le=150, description="Age in years")
    city: str = Field(description="City of residence")
    skills: list[str] = Field(description="List of skills")

person = Person(**person_data)
print(f"✅ Validated: {person}")
print(f"\nJSON Schema: {json.dumps(Person.model_json_schema(), indent=2)}")

try:
    Person(name="Nazar", age="not_a_number", city="Kyiv", skills=[])
except ValidationError as e:
    print(f"\n❌ ValidationError caught:")
    for err in e.errors():
        print(f"   Field '{err['loc'][0]}': {err['msg']}")

✅ Validated: name='Nazar' age=25 city='Kyiv' skills=['Python', 'TypeScript', 'ML']

JSON Schema: {
  "properties": {
    "name": {
      "description": "Person's full name",
      "title": "Name",
      "type": "string"
    },
    "age": {
      "description": "Age in years",
      "maximum": 150,
      "minimum": 0,
      "title": "Age",
      "type": "integer"
    },
    "city": {
      "description": "City of residence",
      "title": "City",
      "type": "string"
    },
    "skills": {
      "description": "List of skills",
      "items": {
        "type": "string"
      },
      "title": "Skills",
      "type": "array"
    }
  },
  "required": [
    "name",
    "age",
    "city",
    "skills"
  ],
  "title": "Person",
  "type": "object"
}

❌ ValidationError caught:
   Field 'age': Input should be a valid integer, unable to parse string as an integer


---

## 3. LangChain — модель-агностичний фреймворк

LangChain — фреймворк, що забезпечує єдиний інтерфейс для роботи з різними LLM-провайдерами. Ключові компоненти:

- **Chat Models** — абстракція над API різних LLM
- **Tools** — декоратор `@tool` для автоматичної генерації JSON-схеми
- **Chains (LCEL)** — декларативні конвеєри через оператор `|`
- **Output Parsers** — перетворення тексту LLM у структуровані дані
- **Memory** — зберігання контексту між викликами
- **Agents** — ReAct цикл (Thought → Action → Observation)


### 3.1 Chat Models — єдиний інтерфейс для різних LLM

Всі моделі мають однаковий метод `.invoke(messages)`. Ключові параметри:
- **model** — назва моделі
- **temperature** — 0.0 (точно) → 1.0 (творчо)
- **streaming** — потоковий вивід


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Same interface, different providers — just swap the class
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

response = llm.invoke([HumanMessage(content="What is 2+2? Answer in one word.")])
print(f"Model:    {llm.model_name}")
print(f"Response: {response.content}")
print(f"Type:     {type(response).__name__}")
print(f"Tokens:   {response.response_metadata.get('token_usage', {})}")

# You can swap to Anthropic with identical .invoke() call:
# from langchain_anthropic import ChatAnthropic
# llm_claude = ChatAnthropic(model="claude-sonnet-4-6")
# response = llm_claude.invoke([HumanMessage(content="What is 2+2?")])

Model:    gpt-5-mini
Response: four
Type:     AIMessage
Tokens:   {'completion_tokens': 74, 'prompt_tokens': 18, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}


### 3.2 Tools — декоратор `@tool`

LangChain читає **ім'я функції**, **docstring** та **type hints** — і автоматично генерує JSON-схему для LLM.

Три обов'язкові елементи:
1. **Ім'я** — назва функції
2. **Docstring** — опис для LLM (ключовий для вибору!)
3. **Type hints** — типи параметрів


In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Return current weather for a city."""
    try:
        resp = requests.get(
            f"https://wttr.in/{city}",
            params={"format": "j1"},
            headers={"User-Agent": "lecture-demo"},
            timeout=5,
        )
        resp.raise_for_status()
        current = resp.json()["current_condition"][0]
        temp_c = current["temp_C"]
        conditions = current["weatherDesc"][0]["value"].lower()
        humidity = current["humidity"]
        return f"{temp_c}°C, {conditions}, humidity {humidity}%"
    except Exception as e:
        return f"Weather data not available for {city}: {e}"

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression and return the result. Example: '2 + 2' -> '4'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {"abs": abs, "round": round, "min": min, "max": max})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Inspect auto-generated schemas
for t in [get_weather, calculate]:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description}")
    print(f"  Schema: {json.dumps(t.args_schema.model_json_schema(), indent=2)}")
    print()

Tool: get_weather
  Description: Return current weather for a city.
  Schema: {
  "description": "Return current weather for a city.",
  "properties": {
    "city": {
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "get_weather",
  "type": "object"
}

Tool: calculate
  Description: Evaluate a math expression and return the result. Example: '2 + 2' -> '4'.
  Schema: {
  "description": "Evaluate a math expression and return the result. Example: '2 + 2' -> '4'.",
  "properties": {
    "expression": {
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "title": "calculate",
  "type": "object"
}



In [ ]:
# Bind tools to model — now LLM can call them
llm_with_tools = llm.bind_tools([get_weather, calculate])

response = llm_with_tools.invoke([HumanMessage(content="What's the weather in Kyiv?")])
print(f"Tool calls: {response.tool_calls}")

Tool calls: [{'name': 'get_weather', 'args': {'city': 'Kyiv'}, 'id': 'call_ETPRQsKw1Er2R6zDHlD8eYiX', 'type': 'tool_call'}]


### 3.3 Chains (LCEL) — конвеєри через оператор `|`

**LCEL (LangChain Expression Language)** — декларативний спосіб будувати конвеєри:

```
dict → prompt → model → parser → result
```

Переваги:
- Автоматичний streaming
- Підтримка `async/await`
- Трасування через LangSmith
- Паралельне виконання


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Build components
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise — answer in 1-2 sentences."),
    ("human", "{question}")
])
model = ChatOpenAI(model="gpt-5-mini", temperature=0)
parser = StrOutputParser()

# 2. Compose via | — this is an LCEL chain
chain = prompt | model | parser

# 3. Invoke with input variables
result = chain.invoke({
    "role": "Python expert",
    "question": "What is a decorator in Python?"
})
print(f"Result (str):\n{result}")
print(f"\nType: {type(result)}")

Result (str):
A decorator is a callable (usually a function) that takes another function or class and returns a new function or class, letting you wrap or modify its behavior without changing its source. It's commonly applied with the @decorator syntax above a definition to add concerns like logging, timing, or access control.

Type: <class 'langchain_core.messages.base.TextAccessor'>


### 3.4 Output Parsers — перетворення тексту на структури

LLM завжди повертає **рядок**. Парсери перетворюють його на те, що потрібно коду:

| Парсер | Результат |
|--------|-----------|
| `StrOutputParser` | `str` |
| `JsonOutputParser` | `dict` |
| `PydanticToolsParser` | Pydantic model |

**Pydantic = валідація**: якщо LLM видасть неправильний формат — отримаємо `ValidationError`, а не тихий баг.


In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser


# StrOutputParser — extracts clean text
chain_str = prompt | model | StrOutputParser()
result_str = chain_str.invoke({"role": "historian", "question": "When was Kyiv founded?"})
print(f"StrOutputParser: {result_str}")
print(f"  Type: {type(result_str)}\n")

# JsonOutputParser — parses JSON from LLM output
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Always respond with valid JSON only. No markdown, no explanation."),
    ("human", "Return a JSON object with 'capital' and 'population' for {country}.")
])

chain_json = json_prompt | model | JsonOutputParser()
result_json = chain_json.invoke({"country": "Ukraine"})

print(f"JsonOutputParser: {result_json}")
print(f"  Type: {type(result_json)}")
print(f"  capital = {result_json.get('capital')}")

StrOutputParser: By the Primary Chronicle Kyiv was traditionally founded in 482 by the brothers Kyi, Shchek and Khoryv and their sister Lybid; archaeological evidence, however, shows habitation from at least the 6th century and that Kyiv became a major urban centre by the late 9th–10th centuries.
  Type: <class 'langchain_core.messages.base.TextAccessor'>

JsonOutputParser: {'capital': 'Kyiv', 'population': 36500000}
  Type: <class 'dict'>
  capital = Kyiv


### 3.5 Pydantic + LLM — structured output через `with_structured_output`

LangChain дозволяє передати Pydantic-модель напряму в LLM через `.with_structured_output()` — модель гарантовано поверне об'єкт потрібної структури.


In [ ]:
from pydantic import BaseModel, Field


class CityInfo(BaseModel):
    """Information about a city."""
    city: str = Field(description="City name")
    country: str = Field(description="Country name")
    population: int = Field(description="Approximate population")
    famous_for: list[str] = Field(description="What the city is famous for, 2-3 items")


# with_structured_output forces the model to return a Pydantic object
structured_llm = model.with_structured_output(CityInfo)

result = structured_llm.invoke("Tell me about Kyiv")
print(f"Type: {type(result).__name__}")
print(f"City: {result.city}")
print(f"Country: {result.country}")
print(f"Population: {result.population:,}")
print(f"Famous for: {result.famous_for}")

Type: CityInfo
City: Kyiv
Country: Ukraine
Population: 2,900,000
Famous for: ['Historic architecture and UNESCO sites (Saint Sophia Cathedral, Kyiv Pechersk Lavra)', 'Center of Ukrainian politics and modern history (Independence Square/Maidan)', 'Dnieper River, parks and rich cultural institutions (museums, theaters)']


---

## 4. Пам'ять та Context Engineering

### Типи пам'яті агента

| Тип | Область | Lifetime | Приклад |
|-----|---------|----------|---------|
| **Working Memory** | Поточна сесія | До кінця запиту | `messages[]`, tool results, system prompt |
| **Short-Term** | Сесія користувача | Тривалість сесії | `BufferMemory`, `SummaryMemory` |
| **Long-Term** | Між сесіями | Постійно | Postgres, Redis (persistent) |
| **Semantic / RAG** | Знання, документи | Постійно | Pinecone, Chroma, pgvector |

> **Context Engineering** — явне керування тим, що потрапляє в контекстне вікно LLM. Це найцінніший ресурс агента.


In [ ]:
# Working Memory — explicitly managing the context window

working_memory = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
]

turns = [
    "My name is Alice and I live in Kyiv.",
    "What city do I live in?",
    "What is 15 * 23?",
]

for user_msg in turns:
    working_memory.append({"role": "user", "content": user_msg})

    response = client.responses.create(
        model="gpt-5-mini",
        input=working_memory,
    )

    assistant_msg = response.output_text
    working_memory.append({"role": "assistant", "content": assistant_msg})
    print(f"👤 {user_msg}")
    print(f"🤖 {assistant_msg}\n")

print(f"Context window: {len(working_memory)} messages")

👤 My name is Alice and I live in Kyiv.
🤖 Hi Alice — nice to meet you. How can I help you today? I can look up local news, safety and travel info, services in Kyiv, translate or draft messages, or anything else. Would you like me to reply in Ukrainian?

👤 What city do I live in?
🤖 You live in Kyiv.

👤 What is 15 * 23?
🤖 15 × 23 = 345

Context window: 7 messages


### InMemorySaver — автоматичне збереження контексту між викликами

В LangChain пам'ять вбудована через `checkpointer`. Кожен `thread_id` зберігає окрему історію розмови.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# Create agent with memory (checkpointer persists state per thread)
checkpointer = InMemorySaver()
agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather, calculate],
    checkpointer=checkpointer,
)

# Thread "alice" — each thread has its own conversation history
config = {"configurable": {"thread_id": "alice"}}

msg1 = "My name is Alice. I live in Kyiv."
print(f"👤 {msg1}")
resp1 = agent.invoke(
    {"messages": [{"role": "user", "content": msg1}]},
    config=config
)
print(f"🤖 {resp1['messages'][-1].content}\n")

# The agent remembers the previous context via checkpointer
msg2 = "What is my name and where do I live?"
print(f"👤 {msg2}")
resp2 = agent.invoke(
    {"messages": [{"role": "user", "content": msg2}]},
    config=config
)
print(f"🤖 {resp2['messages'][-1].content}\n")

# Different thread — separate memory
config_bob = {"configurable": {"thread_id": "bob"}}
msg3 = "What is my name?"
print(f"👤 (bob thread) {msg3}")
resp3 = agent.invoke(
    {"messages": [{"role": "user", "content": msg3}]},
    config=config_bob
)
print(f"🤖 (bob thread) {resp3['messages'][-1].content}")

👤 My name is Alice. I live in Kyiv.
🤖 Hi Alice — nice to meet you. How can I help you today?

I can assist with things related to Kyiv (weather, local news, services, travel, safety tips), translations, writing or editing, planning, or anything else you need. Would you like me to look up the current weather or news for Kyiv now?

👤 What is my name and where do I live?
🤖 Your name is Alice, and you live in Kyiv. Would you like help with anything related to Kyiv or something else?

👤 (bob thread) What is my name?
🤖 (bob thread) I don't know your name — I don't have access to personal info unless you tell me. What would you like me to call you? (You can give a full name, a nickname, or ask me to pick one.)


---

## 5. Agents & ReAct цикл

Агент — це LLM у циклі прийняття рішень:

1. **Thought** — LLM думає, що робити далі
2. **Action** — викликає інструмент
3. **Observation** — отримує результат
4. **Final Answer** — або повторює цикл

### LangChain `create_agent`

Створює повноцінного ReAct-агента з автоматичним циклом виконання.

In [ ]:
# ReAct Agent with tools — full execution loop
agent_no_mem = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather, calculate],
)


# The agent decides which tools to call and chains results
response = agent_no_mem.invoke({
    "messages": [{"role": "user", "content": "What's the weather in London? Also, what is 144 / 12?"}]
})


# Print the full execution trace
for msg in response["messages"]:
    msg_type = type(msg).__name__
    if msg_type == "HumanMessage":
        print(f"👤 Human: {msg.content}")
    elif msg_type == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"🧠 AI calls: {tc['name']}({tc['args']})")
        if msg.content:
            print(f"🤖 AI: {msg.content}")
    elif msg_type == "ToolMessage":
        print(f"🔧 Tool [{msg.name}]: {msg.content}")
    print()

👤 Human: What's the weather in London? Also, what is 144 / 12?

🧠 AI calls: get_weather({'city': 'London'})
🧠 AI calls: calculate({'expression': '144 / 12'})

🔧 Tool [get_weather]: 8°C, overcast, humidity 81%

🔧 Tool [calculate]: 12.0

🤖 AI: Current weather in London: 8°C, overcast, humidity 81%.

144 / 12 = 12.



---

## 6. Advanced Tool Use

### Три просунуті техніки:

| Техніка | Проблема | Рішення |
|---------|----------|---------|
| **Tool Search** | 500+ tools = 50K+ токенів | Векторний пошук Top-K інструментів |
| **Code Calling** | Складні обчислення | Sandbox виконання згенерованого коду |
| **Tool Examples** | JSON Schema замало | Навчання через демонстрацію |

### Tool Search — проблема масштабу

- 10 інструментів ≈ 3K токенів
- 100 інструментів ≈ 30K токенів  
- 500+ інструментів → переповнення контексту

**Рішення:** Індексувати описи інструментів у векторному сховищі та отримувати Top-K за потреби.


In [ ]:
# Tool Search — dynamic tool discovery via vector store
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Register a catalog of tool descriptions
tool_descriptions = [
    "get_weather: Get current weather conditions for any city worldwide",
    "search_web: Search the internet for information on any topic",
    "send_email: Send an email message to a recipient with subject and body",
    "query_database: Execute SQL queries against a relational database",
    "create_calendar_event: Create a new event in the user's calendar",
    "translate_text: Translate text between any two languages",
    "generate_image: Generate an image from a text description using AI",
    "analyze_sentiment: Determine the sentiment (positive/negative/neutral) of text",
    "get_stock_price: Get real-time stock price for a ticker symbol",
    "summarize_document: Create a concise summary of a long document",
    "convert_currency: Convert amounts between different currencies",
    "get_directions: Get driving/walking directions between two locations",
    "create_ticket: Create a support ticket in the issue tracking system",
    "check_inventory: Check product availability and stock levels",
    "schedule_meeting: Find available time slots and schedule a meeting",
]

# Index all tool descriptions
tool_index = Chroma.from_texts(tool_descriptions, embeddings)

# On-demand retrieval: find only relevant tools for a query
queries = [
    "What's the weather in Berlin?",
    "I need to email my colleague about the meeting",
    "How much is 100 USD in EUR?",
    "Show me a picture of a sunset",
]

print(f"Total tools in catalog: {len(tool_descriptions)}")
print(f"Without tool search: ~{len(tool_descriptions) * 300:,} tokens per LLM call")
print(f"With tool search (k=3): ~{3 * 300} tokens per LLM call\n")

for query in queries:
    relevant = tool_index.similarity_search(query, k=3)
    tool_names = [doc.page_content.split(":")[0] for doc in relevant]
    print(f"🔍 '{query}'")
    print(f"   → {tool_names}\n")

Total tools in catalog: 15
Without tool search: ~4,500 tokens per LLM call
With tool search (k=3): ~900 tokens per LLM call

🔍 'What's the weather in Berlin?'
   → ['get_weather', 'get_directions', 'analyze_sentiment']

🔍 'I need to email my colleague about the meeting'
   → ['schedule_meeting', 'send_email', 'create_calendar_event']

🔍 'How much is 100 USD in EUR?'
   → ['convert_currency', 'get_stock_price', 'get_weather']

🔍 'Show me a picture of a sunset'
   → ['generate_image', 'get_weather', 'schedule_meeting']



### Code Calling — виконання у Sandbox

Агент генерує код → виконує його в ізольованому середовищі → перевіряє результат → якщо помилка — виправляє і запускає знову.


In [ ]:
# Code Calling — LLM writes code, sandbox executes, LLM self-corrects on errors

def run_in_sandbox(code: str, timeout: int = 10) -> dict:
    """Execute Python code in a subprocess sandbox.
    In production: Docker container with --network=none, gVisor, resource limits."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout
        )
        return {"stdout": result.stdout.strip(), "stderr": result.stderr.strip(),
                "success": result.returncode == 0}
    except subprocess.TimeoutExpired:
        return {"stdout": "", "stderr": f"Timeout after {timeout}s", "success": False}


# Tool schema for code execution
code_tool = {
    "type": "function",
    "name": "run_python",
    "description": "Execute Python code in a sandbox. Use print() to output the result.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {"type": "string", "description": "Python code to execute"}
        },
        "required": ["code"]
    }
}

SYSTEM_PROMPT = (
    "You are a coding assistant. Solve tasks by writing Python code. "
    "Use the run_python tool. Always print() the final answer. "
    "If execution fails, analyze the error and try again with fixed code."
)

def code_calling_agent(task: str, max_attempts: int = 3) -> str:
    """Full code-calling loop: LLM generates → sandbox executes → LLM self-corrects."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]

    for attempt in range(1, max_attempts + 1):
        # Step 1: LLM generates code via tool call
        response = client.responses.create(
            model="gpt-5-mini",
            input=messages,
            tools=[code_tool],
            tool_choice="required",
        )

        tool_call = next(item for item in response.output if item.type == "function_call")
        code = json.loads(tool_call.arguments)["code"]
        print(f"--- Attempt {attempt} ---")
        print(f"📝 Code:\n{code}\n")

        # Step 2: Execute in sandbox
        result = run_in_sandbox(code)

        if result["success"]:
            print(f"✅ Output: {result['stdout']}")
            return result["stdout"]

        # Step 3: Feed error back to LLM for self-correction
        error_msg = result["stderr"]
        print(f"❌ Error: {error_msg}\n")

        messages.extend(response.output)
        messages.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": f"EXECUTION FAILED:\n{error_msg}\n\nAnalyze the error and write corrected code.",
        })

    return "Max attempts reached"


# --- Demo 1: Straightforward task ---
print("=" * 60)
print("Task 1: Sum of all prime numbers under 100")
print("=" * 60)
code_calling_agent("Compute the sum of all prime numbers under 100. Print only the number.")

print("\n")

# --- Demo 2: Multi-step computation ---
print("=" * 60)
print("Task 2: Statistical measures with the statistics module")
print("=" * 60)
code_calling_agent(
    "Use the 'statistics' module to compute the harmonic mean of [2, 4, 8, 16]. "
    "Also compute the geometric mean of the same list. Print both values rounded to 4 decimal places."
)

Task 1: Sum of all prime numbers under 100
--- Attempt 1 ---
📝 Code:
def is_prime(n):
    if n < 2:
        return False
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True

print(sum(i for i in range(2, 100) if is_prime(i)))


✅ Output: 1060


Task 2: Statistical measures with the statistics module
--- Attempt 1 ---
📝 Code:
from statistics import harmonic_mean, geometric_mean

data = [2, 4, 8, 16]

h = harmonic_mean(data)
 g = geometric_mean(data)
print(f"Harmonic mean: {h:.4f}")
print(f"Geometric mean: {g:.4f}")

❌ Error: File "<string>", line 6
    g = geometric_mean(data)
IndentationError: unexpected indent

--- Attempt 2 ---
📝 Code:
from statistics import harmonic_mean, geometric_mean

data = [2, 4, 8, 16]

h = harmonic_mean(data)
 g = geometric_mean(data)
print(f"Harmonic mean: {h:.4f}")
print(f"Geometric mean: {g:.4f}")

❌ Error: File "<string>", line 6
    g = geometric_mean(data)
IndentationError: unexpected indent

--

'Harmonic mean: 4.2667\nGeometric mean: 5.6569'